# 06 - Transfer Learning InceptionV3

InceptionV3 preentrenada en ImageNet. Mismo flujo que VGG16 y ResNet50: fase 1 con la base congelada y fase 2 de fine-tuning. Como ResNet50, InceptionV3 tiene BatchNormalization en toda la red, asi que paso training=False en ambas fases para que las BN no cambien. Uso 160x160 en lugar de 299x299 para reducir el tiempo de entrenamiento en Colab; con include_top=False el modelo acepta cualquier resolucion mayor a 75px.

> **Nota:** si `inceptionv3_best.keras` ya esta en la carpeta de Drive, gdown lo descarga al inicio. Hay una celda opcional antes de la fase 1 que carga el modelo directamente para saltarse el entrenamiento.

In [ ]:
!pip install tensorflow-datasets gdown scikit-learn seaborn --quiet

In [ ]:
import os

WORK_PATH = '/content/plantvillage'
os.makedirs(WORK_PATH, exist_ok=True)

In [ ]:
import gdown
gdown.download_folder(
    'https://drive.google.com/drive/folders/1OCOyDSR9C3TCzLthsMxJCmy6wcjM3pxs',
    output=WORK_PATH,
    quiet=True
)

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import classification_report, confusion_matrix

# precision mixta para aprovechar los tensor cores de la GPU
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print('tensorflow:', tf.__version__)
print('gpu disponible:', tf.config.list_physical_devices('GPU'))
print('precision global:', tf.keras.mixed_precision.global_policy().name)

tf.random.set_seed(42)
np.random.seed(42)

## Parametros y datos

Uso 160x160 en lugar de 299x299 para reducir el computo; con include_top=False InceptionV3 acepta cualquier resolucion mayor a 75px. Bajo el batch a 64 aprovechando precision mixta.

In [ ]:
with open(f'{WORK_PATH}/dataset_info.json') as f:
    ds_info = json.load(f)
with open(f'{WORK_PATH}/class_weight.json') as f:
    class_weight = {int(k): v for k, v in json.load(f).items()}

NUM_CLASSES      = ds_info['num_classes']
CLASS_NAMES      = ds_info['class_names']
IMG_SIZE         = 160   
BATCH_SIZE       = 64    
STEPS_PER_EPOCH  = 100   
AUTOTUNE         = tf.data.AUTOTUNE

print('clases:', NUM_CLASSES)
print('tamano imagen:', IMG_SIZE)
print('batch size:', BATCH_SIZE)
print('steps por epoch:', STEPS_PER_EPOCH)

In [ ]:
(ds_train_raw, ds_val_raw, ds_test_raw), _ = tfds.load(
    'plant_village',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    with_info=True,
    as_supervised=True,
    shuffle_files=True,
    read_config=tfds.ReadConfig(shuffle_seed=42)
)

# dejo las imagenes en [0, 255]; el preprocesado especifico va dentro del modelo
def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

ds_train = (
    ds_train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .shuffle(2000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_val = (
    ds_val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_test = (
    ds_test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print('pipeline listo')

## Construccion del modelo

Augmentation, preprocesado de InceptionV3 y base congelada. Paso training=False para mantener las BN fijas en ambas fases.

In [ ]:
base_model = tf.keras.applications.InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# augmentation en [0,255]
x = tf.keras.layers.RandomFlip('horizontal_and_vertical')(inputs)
x = tf.keras.layers.RandomRotation(0.2)(x)
x = tf.keras.layers.RandomZoom(0.2)(x)
x = tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255))(x)
x = tf.keras.layers.RandomContrast(0.2)(x)

# preprocesado especifico de InceptionV3: escala de [0,255] a [-1, 1]
x = tf.keras.layers.Lambda(
    tf.keras.applications.inception_v3.preprocess_input,
    name='inceptionv3_preprocess'
)(x)

# training=False para mantener las BN fijas
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

model = tf.keras.Model(inputs, outputs, name='inceptionv3_transfer')

total      = model.count_params()
entrenables = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'parametros totales:      {total:,}')
print(f'parametros entrenables:  {entrenables:,}')

In [ ]:
model.summary()

In [ ]:
# si inceptionv3_best.keras ya fue descargado por gdown al inicio, podes cargar el modelo
# y saltarte las celdas de fase 1, fase 2 y curvas
# cambia REENTRENAR = True si queres volver a entrenar desde cero
REENTRENAR = False

ruta_modelo = f'{WORK_PATH}/inceptionv3_best.keras'

if os.path.exists(ruta_modelo) and not REENTRENAR:
    model = tf.keras.models.load_model(
        ruta_modelo,
        custom_objects={'preprocess_input': tf.keras.applications.inception_v3.preprocess_input}
    )
    print('modelo cargado desde Drive')
    print('saltate las celdas de fase 1, fase 2 y curvas')
    print('ejecuta directamente la celda de evaluacion en test')
else:
    print('modelo no encontrado o REENTRENAR=True, ejecuta las celdas de entrenamiento')

## Fase 1 - feature extraction

Solo se actualiza la cabeza. Learning rate alto porque la base esta completamente congelada. Limito los pasos por epoch para que el entrenamiento sea viable en Colab.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_fase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=4, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f'{WORK_PATH}/inceptionv3_fase1_best.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print('fase 1: feature extraction')
history_fase1 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=10,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weight,
    callbacks=callbacks_fase1
)

## Fase 2 - fine-tuning

Descongelo los bloques mixed9 y mixed10, que son los de mas alto nivel de la red. Afino con learning rate bajo y sigo pasando training=False a la base para que las BN no cambien.

In [ ]:
base_model.trainable = True
# congelo todo excepto los bloques mixed9 y mixed10
for layer in base_model.layers:
    if 'mixed9' not in layer.name and 'mixed10' not in layer.name:
        layer.trainable = False

entrenables_ft = sum(tf.size(w).numpy() for w in model.trainable_weights)
capas_ft = [l.name for l in base_model.layers if l.trainable]
print('capas descongeladas:', capas_ft)
print('parametros entrenables en fase 2:', f'{entrenables_ft:,}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_fase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f'{WORK_PATH}/inceptionv3_best.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    )
]

print('fase 2: fine-tuning')
history_fase2 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=10,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weight,
    callbacks=callbacks_fase2
)

## Curvas de aprendizaje por fase

In [ ]:
def plot_curvas(history, titulo):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history.history['accuracy'],     label='train')
    ax1.plot(history.history['val_accuracy'], label='val')
    ax1.set_title(f'{titulo} - accuracy')
    ax1.set_xlabel('epoca')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(history.history['loss'],     label='train')
    ax2.plot(history.history['val_loss'], label='val')
    ax2.set_title(f'{titulo} - loss')
    ax2.set_xlabel('epoca')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{WORK_PATH}/inceptionv3_curvas_{titulo}.png', dpi=150)
    plt.show()

plot_curvas(history_fase1, 'fase1')
plot_curvas(history_fase2, 'fase2')

## Curva combinada fase 1 + fase 2

Uno los historiales de las dos fases para ver el entrenamiento completo en una sola grafica.

In [ ]:
acc_total   = history_fase1.history['accuracy']     + history_fase2.history['accuracy']
val_total   = history_fase1.history['val_accuracy'] + history_fase2.history['val_accuracy']
loss_total  = history_fase1.history['loss']         + history_fase2.history['loss']
vloss_total = history_fase1.history['val_loss']     + history_fase2.history['val_loss']

corte = len(history_fase1.history['loss'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(acc_total,  label='train')
ax1.plot(val_total,  label='val')
ax1.axvline(corte, color='gray', linestyle='--', alpha=0.7, label='inicio fine-tuning')
ax1.set_title('accuracy completo')
ax1.set_xlabel('epoca')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(loss_total,  label='train')
ax2.plot(vloss_total, label='val')
ax2.axvline(corte, color='gray', linestyle='--', alpha=0.7, label='inicio fine-tuning')
ax2.set_title('loss completo')
ax2.set_xlabel('epoca')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('InceptionV3 - trayectoria completa de entrenamiento', fontsize=12)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/inceptionv3_curvas_completo.png', dpi=150)
plt.show()

## Evaluacion en test

In [ ]:
best_model = tf.keras.models.load_model(
    f'{WORK_PATH}/inceptionv3_best.keras',
    custom_objects={'preprocess_input': tf.keras.applications.inception_v3.preprocess_input}
)

# recorro el dataset una vez para que pred y labels queden alineados
y_pred_list, y_true_list = [], []
for images, labels in ds_test:
    preds = best_model(images, training=False)
    y_pred_list.extend(np.argmax(preds.numpy(), axis=1))
    y_true_list.extend(labels.numpy())

y_pred = np.array(y_pred_list)
y_true = np.array(y_true_list)

test_acc = np.mean(y_pred == y_true)
print(f'accuracy en test: {test_acc:.4f} ({test_acc*100:.2f}%)')
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize='true')

plt.figure(figsize=(22, 20))
sns.heatmap(cm, annot=False, cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('matriz de confusion normalizada - InceptionV3', fontsize=13)
plt.xlabel('prediccion')
plt.ylabel('real')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/inceptionv3_confusion.png', dpi=150)
plt.show()

In [ ]:
resultados_inceptionv3 = {
    'modelo'       : 'InceptionV3',
    'test_acc'     : float(test_acc),
    'epochs_fase1' : len(history_fase1.history['loss']),
    'epochs_fase2' : len(history_fase2.history['loss'])
}

with open(f'{WORK_PATH}/resultados_inceptionv3.json', 'w') as f:
    json.dump(resultados_inceptionv3, f, indent=2)

print('resultados guardados')